# An agent workflow with an approval

A [Temporal](https://temporal.io/) workflow asks a prokube Agent over A2A to research a topic, waits until a person approves the result, then asks the Agent for a summary. The workflow survives restarts: while it waits, nothing runs, and the approval can come days later from this notebook or from the prokube UI.

You need a prokube Agent in your workspace and an API key with access to it; see the [README](./README.md).

## Setup

In [ ]:
%pip install -q "prokube-sdk[workflows] @ git+https://github.com/prokube/prokube-sdk@feature/workflows-keycloak-login"

In [ ]:
import os
from getpass import getpass
from pathlib import Path

PLATFORM = "https://<your-prokube-domain>"  # change this
WORKSPACE = Path("/var/run/secrets/kubernetes.io/serviceaccount/namespace").read_text()
AGENT = "research-assistant"  # an Agent in this workspace
TASK_QUEUE = "agent-approval"

os.environ["A2A_URL"] = f"{PLATFORM}/svc/a2a/{WORKSPACE}"
os.environ["A2A_API_KEY"] = getpass("prokube API key with access to the Agent: ")

## Log in

Temporal checks your own login. In a Lab, the login prints a URL and a code: open the URL in your browser and confirm the code.

In [ ]:
from prokube import auth

auth.login(PLATFORM, provider=auth.WORKFLOWS)

## Run a worker

The worker runs the workflow code from [`workflows.py`](./workflows.py) and the A2A call from [`activities.py`](./activities.py). Here it runs in the background of this notebook; a worker that always runs is a container in the workspace.

In [ ]:
import asyncio

from prokube.workflows import connect
from temporalio.worker import Worker

from activities import send_a2a_message
from workflows import ApprovalDecision, ResearchRequest, ResearchWithApproval

client = await connect(PLATFORM, WORKSPACE)
worker = Worker(client, task_queue=TASK_QUEUE, workflows=[ResearchWithApproval], activities=[send_a2a_message])
worker_task = asyncio.create_task(worker.run())

## Start a run

In [ ]:
import uuid

handle = await client.start_workflow(
    ResearchWithApproval.run,
    ResearchRequest(topic="How do sea otters use tools?", agent=AGENT),
    id=f"agent-approval-{uuid.uuid4().hex[:6]}",
    task_queue=TASK_QUEUE,
)
print(f"Started {handle.id}. It also shows in the prokube UI under Workflows.")

## Approve

The run waits for a person twice: when the Agent wants to run a tool, and before the summary. Answer each question here with `y` or `n`, or open the run under **Workflows** in the prokube UI and answer it there.

In [ ]:
while (await handle.describe()).status.name == "RUNNING":
    pending = await handle.query(ResearchWithApproval.pending_approval)
    if pending is None:
        await asyncio.sleep(2)
        continue
    approved = input(f"{pending.question}\n\nApprove? [y/n] ").strip().lower() == "y"
    await handle.execute_update(
        ResearchWithApproval.approve,
        ApprovalDecision(approved=approved, approver="notebook", approval_id=pending.id),
    )

In [ ]:
result = await handle.result()
print(result.summary if result.approved else "Rejected.")

## Stop the worker

In [ ]:
await worker.shutdown()